# AIAP Mini Project 02: Hotel Customer No-Show Prediction

**Dataset:** `mini_project_2_data.csv`  
**Task:** Binary Classification (`no_show`: `0` = Show, `1` = No-Show)  
**Reference:** *AIAP Foundation Learning Companion* — Section 1.2 (EDA) & Section 1.3 (Data Cleaning and Preprocessing)

---

In [ ]:
# Imports and Visual Style Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Constants
SGD_TO_USD_RATE = 1.28  # Constant: 1 USD = 1.28 SGD
RED_BOLD = "\033[1;31m"
GREEN_BOLD = "\033[1;32m"
CLEAR = "\033[0m"

print(f"{GREEN_BOLD}[INFO] Libraries loaded and constants set successfully.{CLEAR}")

## 1. Preliminary Data Exploration & Targeted Data Cleaning

Examine raw data shapes, handle missing labels, convert prices to USD using `SGD_TO_USD_RATE`, correct corrupted checkout days, calculate cross-month `stay_duration`, standardise text numbers (`'one'`, `'two'`), and impute missing values .

In [ ]:
# Load raw dataset
df_raw = pd.read_csv('./data/mini_project_2_data.csv')
print(f"{GREEN_BOLD}[INFO] Raw Dataset Shape: {df_raw.shape}{CLEAR}")

# Drop row with missing target variable ('no_show')
df = df_raw.dropna(subset=['no_show']).copy().reset_index(drop=True)
df['no_show'] = df['no_show'].astype(int)

print(f"{GREEN_BOLD}[INFO] Cleaned Dataset Shape (Target drop applied): {df.shape}{CLEAR}")
df.head()

In [ ]:
# Price Parsing and Conversion to USD
def parse_price_to_usd(raw_price):
    if pd.isna(raw_price):
        return np.nan
    p_str = str(raw_price).strip()
    if 'SGD$' in p_str:
        return float(p_str.replace('SGD$', '').strip()) / SGD_TO_USD_RATE
    elif 'USD$' in p_str:
        return float(p_str.replace('USD$', '').strip())
    else:
        try:
            return float(p_str)
        except:
            return np.nan

df['price_usd'] = df['price'].apply(parse_price_to_usd)

# Standardise text numbers in num_adults / num_children (e.g. 'one' -> 1, 'two' -> 2)
num_str_map = {'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5}
df['num_adults'] = df['num_adults'].replace(num_str_map).astype(float)
df['num_children'] = df['num_children'].replace(num_str_map).astype(float)

# Handle missing room types and grouped price imputation
df['room'] = df['room'].fillna('Unknown')
price_medians = df.groupby(['branch', 'room'])['price_usd'].transform('median')
df['price_usd'] = df['price_usd'].fillna(price_medians)

# Correct negative checkout days and calculate stay duration (accounting for crossover months)
df['checkout_day_clean'] = df['checkout_day'].abs()
days_in_month = {'January':31, 'February':28, 'March':31, 'April':30, 'May':31, 'June':30,
                 'July':31, 'August':31, 'September':30, 'October':31, 'November':30, 'December':31}

def compute_stay_duration(row):
    a_m, c_m = row['arrival_month'], row['checkout_month']
    a_d, c_d = row['arrival_day'], row['checkout_day_clean']
    if pd.isna(a_m) or pd.isna(c_m) or pd.isna(a_d) or pd.isna(c_d):
        return np.nan
    if a_m == c_m:
        return c_d - a_d
    else:
        return (days_in_month.get(a_m, 30) - a_d) + c_d

df['stay_duration'] = df.apply(compute_stay_duration, axis=1)
df.drop(columns=['price', 'checkout_day_clean'], inplace=True)

print(f"{GREEN_BOLD}[INFO] Preprocessing complete. Total Nulls Remaining: {df.isnull().sum().sum()}{CLEAR}")
df[['branch', 'room', 'price_usd', 'stay_duration', 'no_show']].head()

## 2. Univariate Analysis

Visualizing the target class distribution (`no_show`) and key predictors like `price_usd` and `stay_duration` .

In [ ]:
# Target Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(data=df, x='no_show', hue='no_show', ax=axes[0], palette='viridis', legend=False)
axes[0].set_title('Target Class Count (0 = Show, 1 = No-Show)')
axes[0].set_xticklabels(['Show (0)', 'No-Show (1)'])

sns.histplot(df['price_usd'], bins=40, kde=True, ax=axes[1], color='teal')
axes[1].set_title('Cleaned Room Price (USD) Distribution')

plt.tight_layout()
plt.show()

## 3. Data Splitting & Preprocessing Pipeline

Splitting data into Train (80%), Validation (10%), and Test (10%) sets **before** applying `ColumnTransformer` to eliminate data leakage .

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Drop identifier
X = df.drop(columns=['booking_id', 'no_show'])
y = df['no_show']

# Stratified Train / Val / Test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"{GREEN_BOLD}[INFO] Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}{CLEAR}")

# Define feature groups
num_cols = ['price_usd', 'stay_duration', 'num_adults', 'num_children', 'arrival_day', 'checkout_day']
cat_cols = ['branch', 'booking_month', 'arrival_month', 'checkout_month', 'country', 'first_time', 'room', 'platform']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

print(f"{GREEN_BOLD}[INFO] Preprocessor setup complete.{CLEAR}")

## 4. Model Training & Baseline Evaluation

Evaluating 3 candidate classification models: **Logistic Regression**, **K-Nearest Neighbors (KNN)**, and **Decision Tree Classifier** .

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN Classifier (k=5)': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42)
}

results = []
for name, clf in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_val)
    y_prob = pipe.predict_proba(X_val)[:, 1]
    
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    results.append({
        'Model': name,
        'Accuracy': f'{acc:.4f}',
        'Precision': f'{prec:.4f}',
        'Recall': f'{rec:.4f}',
        'F1-Score': f'{f1:.4f}',
        'ROC-AUC': f'{auc:.4f}'
    })

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
print(f"{GREEN_BOLD}[INFO] Baseline Classification Model Results (Validation Set):{CLEAR}")
results_df